# 04 Wikipedia Web Scraping

## Purpose

Use Wikipedia city pages as the required web scraping source. Store raw HTML in Bronze, parse contextual city metadata, validate joinability with `city_id`, and write Silver Parquet.


## Inputs

- `data/silver/city_reference.parquet`
- Wikipedia city URLs from the city reference table or derived from city names.


## Outputs

- `data/bronze/wikipedia_html/<city_id>.html`
- `data/silver/city_metadata.parquet`


## Technologies used

Python, requests, BeautifulSoup, lxml, pandas, Parquet.


## Configuration

Set `RUN_WIKIPEDIA_FETCH=true` to fetch raw HTML. Request behavior uses a clear User-Agent, timeout and small delay. Generated HTML and Parquet outputs are ignored by Git.


In [ ]:
from pathlib import Path
import os
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
DATA_DIR = Path(os.getenv("DATA_DIR", "data"))
CHECKPOINT_DIR = Path(os.getenv("CHECKPOINT_DIR", "data/checkpoints"))


## Implementation

### Input contract

Phase 4 depends on the Phase 2 city reference. If the file has not been generated yet, the notebook includes the same controlled city list as a fallback so the notebook remains understandable, but the normal path is to run notebook `02` first.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import re
import time
import pandas as pd

CITY_REFERENCE_PATH = DATA_DIR / "silver" / "city_reference.parquet"
WIKIPEDIA_RAW_DIR = DATA_DIR / "bronze" / "wikipedia_html"
SILVER_DIR = DATA_DIR / "silver"
WIKIPEDIA_RAW_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)

RUN_WIKIPEDIA_FETCH = os.getenv("RUN_WIKIPEDIA_FETCH", "true").lower() == "true"
REQUEST_SLEEP_SECONDS = float(os.getenv("WIKIPEDIA_REQUEST_SLEEP_SECONDS", "0.5"))
HEADERS = {"User-Agent": "euro-air-quality-pipeline/1.0 educational web scraping project"}

if CITY_REFERENCE_PATH.exists():
    city_reference_df = pd.read_parquet(CITY_REFERENCE_PATH)
else:
    city_reference_df = pd.DataFrame([
        {"city_id": "vienna_at", "city_name": "Vienna", "country_code": "AT", "latitude": 48.2082, "longitude": 16.3738, "wikipedia_url": "https://en.wikipedia.org/wiki/Vienna"},
        {"city_id": "berlin_de", "city_name": "Berlin", "country_code": "DE", "latitude": 52.5200, "longitude": 13.4050, "wikipedia_url": "https://en.wikipedia.org/wiki/Berlin"},
        {"city_id": "paris_fr", "city_name": "Paris", "country_code": "FR", "latitude": 48.8566, "longitude": 2.3522, "wikipedia_url": "https://en.wikipedia.org/wiki/Paris"},
        {"city_id": "madrid_es", "city_name": "Madrid", "country_code": "ES", "latitude": 40.4168, "longitude": -3.7038, "wikipedia_url": "https://en.wikipedia.org/wiki/Madrid"},
        {"city_id": "rome_it", "city_name": "Rome", "country_code": "IT", "latitude": 41.9028, "longitude": 12.4964, "wikipedia_url": "https://en.wikipedia.org/wiki/Rome"},
        {"city_id": "amsterdam_nl", "city_name": "Amsterdam", "country_code": "NL", "latitude": 52.3676, "longitude": 4.9041, "wikipedia_url": "https://en.wikipedia.org/wiki/Amsterdam"},
        {"city_id": "warsaw_pl", "city_name": "Warsaw", "country_code": "PL", "latitude": 52.2297, "longitude": 21.0122, "wikipedia_url": "https://en.wikipedia.org/wiki/Warsaw"},
        {"city_id": "prague_cz", "city_name": "Prague", "country_code": "CZ", "latitude": 50.0755, "longitude": 14.4378, "wikipedia_url": "https://en.wikipedia.org/wiki/Prague"},
    ])

required_city_columns = {"city_id", "city_name", "country_code", "latitude", "longitude"}
missing_city_columns = required_city_columns - set(city_reference_df.columns)
assert not missing_city_columns, f"Missing city reference columns: {missing_city_columns}"
assert city_reference_df["city_id"].is_unique


### Fetch raw HTML into Bronze

The raw HTML archive preserves source evidence before parsing. Failed downloads are recorded rather than silently ignored.


In [ ]:
def wikipedia_url(row: pd.Series) -> str:
    if "wikipedia_url" in row and pd.notna(row["wikipedia_url"]):
        return row["wikipedia_url"]
    return "https://en.wikipedia.org/wiki/" + str(row["city_name"]).replace(" ", "_")


def fetch_html(url: str, timeout: int = 20) -> tuple[str | None, int | None, str | None]:
    import requests

    try:
        response = requests.get(url, headers=HEADERS, timeout=timeout)
        status_code = response.status_code
        response.raise_for_status()
        return response.text, status_code, None
    except Exception as exc:
        return None, None, str(exc)


retrieval_results = []
for _, row in city_reference_df.iterrows():
    url = wikipedia_url(row)
    output_path = WIKIPEDIA_RAW_DIR / f"{row['city_id']}.html"
    html = None
    status_code = None
    error = None
    status = "skipped"

    if RUN_WIKIPEDIA_FETCH:
        html, status_code, error = fetch_html(url)
        if html:
            output_path.write_text(html, encoding="utf-8")
            status = "success"
        else:
            status = "failed"
        time.sleep(REQUEST_SLEEP_SECONDS)
    elif output_path.exists():
        status = "existing_local_file"

    retrieval_results.append({
        "city_id": row["city_id"],
        "city_name": row["city_name"],
        "url": url,
        "status": status,
        "http_status_code": status_code,
        "file_path": str(output_path) if output_path.exists() else None,
        "file_size_bytes": output_path.stat().st_size if output_path.exists() else None,
        "error": error,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    })

retrieval_results_df = pd.DataFrame(retrieval_results)
retrieval_results_df


### Parse city metadata

The parser is defensive. It attempts to extract population, area and density from the infobox, but records `parse_status` and `parse_notes` because Wikipedia pages differ between cities.


In [ ]:
def clean_number(value: str | None):
    if value is None:
        return None
    text = re.sub(r"\[.*?\]", "", str(value))
    text = text.replace(",", "").replace("\xa0", " ")
    match = re.search(r"([0-9]+(?:\.[0-9]+)?)", text)
    if not match:
        return None
    number = float(match.group(1))
    return int(number) if number.is_integer() else number


def normalize_label(value: str) -> str:
    return re.sub(r"\s+", " ", value.replace("•", " ")).strip().lower()


def find_section_value(soup, section_label: str, preferred_labels: list[str]) -> str | None:
    """Return the city-level value from an infobox section, not a later unrelated row."""
    infobox = soup.select_one("table.infobox")
    if infobox is None:
        return None
    rows = infobox.select("tr")
    section_start = None
    for index, row in enumerate(rows):
        header = row.select_one("th.infobox-header")
        if header and normalize_label(header.get_text(" ", strip=True)).startswith(section_label):
            section_start = index + 1
            break
    if section_start is None:
        return None
    section_rows = []
    for row in rows[section_start:]:
        if row.select_one("th.infobox-header"):
            break
        section_rows.append(row.get_text(" ", strip=True))
    for label in preferred_labels:
        for text in section_rows:
            if normalize_label(text).startswith(label):
                return text
    return None


def find_direct_infobox_value(soup, label: str) -> str | None:
    """Handle compact infobox variants where a value is stored directly on its labeled row."""
    infobox = soup.select_one("table.infobox")
    if infobox is None:
        return None
    for row in infobox.select("tr"):
        header = row.find("th")
        value = row.find("td")
        if header and value and normalize_label(header.get_text(" ", strip=True)).startswith(label):
            return value.get_text(" ", strip=True)
    return None


def parse_city_metadata(city_row: pd.Series, html: str, source_url: str) -> dict:
    from bs4 import BeautifulSoup

    soup = BeautifulSoup(html, "html.parser")
    city_level_labels = ["total", "city/state", "capital city and municipality", "municipality", "capital city and county", "capital city"]
    population_text = find_section_value(soup, "population", city_level_labels) or find_direct_infobox_value(soup, "population")
    area_text = find_section_value(soup, "area", city_level_labels) or find_direct_infobox_value(soup, "area")
    density_text = find_section_value(soup, "population", ["density"]) or find_direct_infobox_value(soup, "density")

    population = clean_number(population_text)
    area_km2 = clean_number(area_text)
    population_density = clean_number(density_text)
    if population_density is None and population and area_km2:
        population_density = population / area_km2

    parsed_values = [population, area_km2, population_density]
    if all(value is not None for value in parsed_values):
        parse_status = "success"
    elif any(value is not None for value in parsed_values):
        parse_status = "partial"
    else:
        parse_status = "failed"

    return {
        "city_id": city_row["city_id"],
        "city_name": city_row["city_name"],
        "country_code": city_row["country_code"],
        "population": population,
        "area_km2": area_km2,
        "population_density": population_density,
        "source_url": source_url,
        "metadata_source": "wikipedia",
        "processed_at_utc": datetime.now(timezone.utc).isoformat(),
        "parse_status": parse_status,
        "parse_notes": "Parsed from Wikipedia infobox with defensive heuristics; values require contextual interpretation.",
    }


metadata_records = []
for _, row in city_reference_df.iterrows():
    path = WIKIPEDIA_RAW_DIR / f"{row['city_id']}.html"
    url = wikipedia_url(row)
    if path.exists():
        metadata_records.append(parse_city_metadata(row, path.read_text(encoding="utf-8"), url))
    else:
        metadata_records.append({
            "city_id": row["city_id"],
            "city_name": row["city_name"],
            "country_code": row["country_code"],
            "population": None,
            "area_km2": None,
            "population_density": None,
            "source_url": url,
            "metadata_source": "wikipedia",
            "processed_at_utc": datetime.now(timezone.utc).isoformat(),
            "parse_status": "failed",
            "parse_notes": "No local raw HTML file available. Run with RUN_WIKIPEDIA_FETCH=true.",
        })

city_metadata_df = pd.DataFrame(metadata_records)
city_metadata_df


## Validation / Quality Checks

Validate one row per `city_id`, non-null join keys, metadata source, parse status, joinability with city reference, and Parquet read-back.


In [ ]:
required_metadata_columns = {
    "city_id", "city_name", "country_code", "population", "area_km2",
    "population_density", "source_url", "metadata_source", "processed_at_utc",
    "parse_status", "parse_notes",
}
assert required_metadata_columns.issubset(city_metadata_df.columns)
assert city_metadata_df["city_id"].notna().all()
assert city_metadata_df["city_id"].is_unique
assert (city_metadata_df["metadata_source"] == "wikipedia").all()
assert set(city_metadata_df["parse_status"]).issubset({"success", "partial", "failed"})
assert city_metadata_df["population"].dropna().gt(0).all()
assert city_metadata_df["area_km2"].dropna().gt(0).all()
assert city_metadata_df["population_density"].dropna().gt(0).all()

joined = city_reference_df.merge(city_metadata_df, on="city_id", how="left", indicator=True)
assert (joined["_merge"] == "both").all()

output_path = SILVER_DIR / "city_metadata.parquet"
city_metadata_df.to_parquet(output_path, index=False)
roundtrip = pd.read_parquet(output_path)
assert len(roundtrip) == len(city_metadata_df)
roundtrip[["city_id", "population", "area_km2", "population_density", "parse_status"]]


## Results

Phase 4 produces a Silver city metadata dataset keyed by `city_id`. Raw HTML is retained locally in Bronze for traceability.


## Limitations

Wikipedia values are contextual, not official ground truth. Page structures and administrative definitions differ. Missing or ambiguous values remain null and are documented through `parse_status` and `parse_notes`. These fields must not be interpreted as causal explanations for air quality.


## Next step

Run notebook `05_open_meteo_api_and_kafka_producer.ipynb` to implement the REST API and Kafka producer path.
